In [1]:
from google.colab import drive
drive.mount('/content/drive')

#!pip install torch torchvision torchaudio
!pip install matplotlib numpy pillow scikit-image opencv-python pandas seaborn imageio
!pip install -q imageio-ffmpeg

Mounted at /content/drive


In [2]:
import sys
import os
sys.path.append('/content/drive/MyDrive/ResearchProject')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import time

from unet_denoiser_v2 import BlindVideoDenoiserUNet
from video_io import VideoLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

project_dir = Path('/content/drive/MyDrive/ResearchProject')
benchmark_dir = project_dir / 'benchmark_results_sr'
benchmark_dir.mkdir(exist_ok=True)

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 42.4 GB


In [3]:
from inverse_problem_framework import LinearOperator, VideoDenoiser, KadkhodaieSolver
from linear_operators import SuperResolutionOperator, create_sr_degradation
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim

print("✓ Framework imported")

✓ Framework imported


In [4]:
class BlindVideoDenoiserWrapper(VideoDenoiser):
    def __init__(self, model, device='cuda', pad_to=32, num_input_frames=5):
        self.model = model.to(device)
        self.device = device
        self.pad_to = pad_to
        self.num_input_frames = num_input_frames
        self.half_window = num_input_frames // 2
        self.model.eval()

    def _pad(self, frame):
        C, H, W = frame.shape
        ph = (self.pad_to - H % self.pad_to) % self.pad_to
        pw = (self.pad_to - W % self.pad_to) % self.pad_to
        if ph > 0 or pw > 0:
            frame = F.pad(frame, (0, pw, 0, ph), mode='reflect')
        return frame, (H, W)

    def _unpad(self, frame, orig):
        return frame[:, :orig[0], :orig[1]]

    def denoise(self, noisy_video, noise_std=0.0):
        T, C, H, W = noisy_video.shape
        noisy_video = noisy_video.to(self.device)
        frames = []
        with torch.no_grad():
            for t in range(T):
                indices = [max(0, min(T-1, t+off)) for off in range(-self.half_window, self.half_window+1)]
                neighbor_frames = [noisy_video[i] for i in indices]

                padded = []
                orig_size = None
                for f in neighbor_frames:
                    pf, orig_size = self._pad(f)
                    padded.append(pf)

                concat = torch.cat(padded, dim=0).unsqueeze(0)
                out = self.model(concat)
                denoised = self._unpad(out.squeeze(0), orig_size)
                frames.append(torch.clamp(denoised, 0, 1))
        return torch.stack(frames)

    def denoise_frame(self, prev_frame, curr_frame, next_frame, noise_std=0.0):
        video = torch.stack([prev_frame, prev_frame, curr_frame, next_frame, next_frame])
        return self.denoise(video, noise_std)[2]

print("✓ Denoiser wrapper defined (5-frame, pad_to=32)")

✓ Denoiser wrapper defined (5-frame, pad_to=32)


In [5]:
print("Loading 5-frame blind video denoiser...")
from unet_denoiser_v2 import BlindVideoDenoiserUNet

checkpoint_path = project_dir / 'checkpoints_large_model' / 'best_model_final.pt'

model = BlindVideoDenoiserUNet(num_input_frames=5, out_channels=3, base_channels=64)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✓ Loaded from epoch {checkpoint['epoch']}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

your_denoiser = BlindVideoDenoiserWrapper(model, device=device, pad_to=32, num_input_frames=5)
print("✓ Wrapped (5 frames, pad_to=32)")

Loading 5-frame blind video denoiser...
✓ Loaded from epoch 26
  Parameters: 18,693,504
✓ Wrapped (5 frames, pad_to=32)


In [6]:
print("Loading test videos...")
davis_root = Path('/content/drive/MyDrive/ResearchProject/DAVISDataset')
test_videos = {}
video_dirs = sorted([d for d in davis_root.iterdir() if d.is_dir()])[:5]

for vdir in video_dirs:
    name = vdir.name
    print(f"  Loading {name}...", end=' ', flush=True)
    try:
        v = VideoLoader.load_frame_sequence(
            str(vdir), max_frames=None, resize=(512, 832), device=device
        )
        test_videos[name] = v
        print(f"✓ {v.shape}")
    except Exception as e:
        print(f"✗ ({e})")

print(f"\nLoaded {len(test_videos)} test videos")

Loading test videos...
  Loading baseball... Loaded 90 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/baseball
  Shape: torch.Size([90, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([90, 3, 512, 832])
  Loading basketball-game... Loaded 77 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/basketball-game
  Shape: torch.Size([77, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([77, 3, 512, 832])
  Loading bear... Loaded 82 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bear
  Shape: torch.Size([82, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([82, 3, 512, 832])
  Loading bears-ball... Loaded 78 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bears-ball
  Shape: torch.Size([78, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([78, 3, 512, 832])
  Loading bike-packing... Loaded 69 frames from: /content/drive/MyDrive/ResearchProject/DAVISDataset/bike-packing
  Shape: torch.Size([69, 3, 512, 832]) (T, C, H, W)
✓ torch.Size([69, 3, 512, 832])

Loaded 5

In [7]:
print("=" * 60)
print("SUPER-RESOLUTION TEST (2x)")
print("=" * 60)

scale_factor = 2
all_sr_results = {}

for video_name, clean in test_videos.items():
    T, C, H, W = clean.shape
    print(f"\n--- {video_name} ({T} frames, {W}×{H}) ---")

    # Create SR degradation
    operator, lr_video, lr_upsampled, bilinear_upsampled = create_sr_degradation(
        clean, scale_factor=scale_factor, device=device
    )
    print(f"  Clean: {clean.shape} → LR: {lr_video.shape} → Upsampled: {lr_upsampled.shape}")

    # The solver needs the LOW-RES observation as input.
    # It will internally upsample via adjoint() to work at full resolution.
    solver = KadkhodaieSolver(operator, your_denoiser, device=device)

    print("  Solving...")
    restored, metrics = solver.solve(
        lr_video,
        sigma_0=0.25,       # SR needs slightly more initial noise than demosaicing
        sigma_L=0.005,      # Fine detail threshold
        h0=0.01,
        beta=0.1,
        max_iterations=2000,
        verbose=True,
        log_freq=200,
    )

    all_sr_results[video_name] = {
        'clean': clean,
        'lr_video': lr_video,
        'lr_upsampled': lr_upsampled,
        'bilinear_upsampled': bilinear_upsampled,
        'restored': restored,
        'metrics': metrics,
    }

    # Quick summary
    mid = T // 2
    c = clean[mid].permute(1, 2, 0).cpu().numpy()
    nn_np = lr_upsampled[mid].permute(1, 2, 0).cpu().numpy()
    bi_np = bilinear_upsampled[mid].permute(1, 2, 0).cpu().numpy()
    r = restored[mid].permute(1, 2, 0).cpu().numpy()
    print(f"  Nearest:  {compute_psnr(c, np.clip(nn_np,0,1), data_range=1.0):.2f} dB")
    print(f"  Bilinear: {compute_psnr(c, np.clip(bi_np,0,1), data_range=1.0):.2f} dB")
    print(f"  Solver:   {compute_psnr(c, np.clip(r,0,1), data_range=1.0):.2f} dB")

print(f"\n{'=' * 60}")
print(f"Done — {len(all_sr_results)} videos processed")
print(f"{'=' * 60}")

SUPER-RESOLUTION TEST (2x)

--- baseball (90 frames, 832×512) ---
  Clean: torch.Size([90, 3, 512, 832]) → LR: torch.Size([90, 3, 256, 416]) → Upsampled: torch.Size([90, 3, 512, 832])
  Solving...
  Converged in 89 iters (141.2s), final sigma=0.004974
  Nearest:  28.14 dB
  Bilinear: 28.27 dB
  Solver:   29.10 dB

--- basketball-game (77 frames, 832×512) ---
  Clean: torch.Size([77, 3, 512, 832]) → LR: torch.Size([77, 3, 256, 416]) → Upsampled: torch.Size([77, 3, 512, 832])
  Solving...
  Converged in 93 iters (125.5s), final sigma=0.004632
  Nearest:  27.34 dB
  Bilinear: 27.28 dB
  Solver:   29.68 dB

--- bear (82 frames, 832×512) ---
  Clean: torch.Size([82, 3, 512, 832]) → LR: torch.Size([82, 3, 256, 416]) → Upsampled: torch.Size([82, 3, 512, 832])
  Solving...
  Converged in 79 iters (113.6s), final sigma=0.004863
  Nearest:  27.15 dB
  Bilinear: 27.18 dB
  Solver:   27.09 dB

--- bears-ball (78 frames, 832×512) ---
  Clean: torch.Size([78, 3, 512, 832]) → LR: torch.Size([78, 3, 2

In [8]:
frames_per_video = 6

for video_name, data in all_sr_results.items():
    clean = data['clean']
    lr_upsampled = data['lr_upsampled']
    bilinear = data['bilinear_upsampled']
    restored = data['restored']
    T = clean.shape[0]

    step = max(1, T // frames_per_video)
    frame_indices = list(range(0, T, step))[:frames_per_video]

    print(f"\n--- {video_name} ({T} frames) ---")

    for idx in frame_indices:
        c_np = clean[idx].permute(1, 2, 0).cpu().numpy()
        lr_np = lr_upsampled[idx].permute(1, 2, 0).cpu().numpy()
        b_np = bilinear[idx].permute(1, 2, 0).cpu().numpy()
        r_np = restored[idx].permute(1, 2, 0).cpu().numpy()

        p_lr = compute_psnr(c_np, np.clip(lr_np, 0, 1), data_range=1.0)
        p_b = compute_psnr(c_np, np.clip(b_np, 0, 1), data_range=1.0)
        p_r = compute_psnr(c_np, np.clip(r_np, 0, 1), data_range=1.0)

        fig, axes = plt.subplots(2, 2, figsize=(18, 14))

        axes[0, 0].imshow(np.clip(c_np, 0, 1))
        axes[0, 0].set_title('Clean (Ground Truth)', fontsize=13, fontweight='bold')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(np.clip(lr_np, 0, 1))
        axes[0, 1].set_title(f'Degraded (Nearest Upsample) — PSNR={p_lr:.1f} dB',
                             fontsize=13, fontweight='bold')
        axes[0, 1].axis('off')

        axes[1, 0].imshow(np.clip(b_np, 0, 1))
        axes[1, 0].set_title(f'Bilinear Upsample — PSNR={p_b:.1f} dB',
                             fontsize=13, fontweight='bold')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(np.clip(r_np, 0, 1))
        axes[1, 1].set_title(f'Solver Output — PSNR={p_r:.1f} dB',
                             fontsize=13, fontweight='bold')
        axes[1, 1].axis('off')

        plt.suptitle(f'{video_name} — Frame {idx+1}/{T} (SR {scale_factor}x)',
                     fontsize=15, fontweight='bold')
        plt.tight_layout()
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# Cell 7b: Save 6 frames per video as PNG images
from PIL import Image

frames_per_video = 6
save_root = str(benchmark_dir / 'sr_frames')

for video_name, data in all_sr_results.items():
    clean = data['clean']
    lr_upsampled = data['lr_upsampled']
    bilinear = data['bilinear_upsampled']
    restored = data['restored']
    T = clean.shape[0]

    step = max(1, T // frames_per_video)
    frame_indices = list(range(0, T, step))[:frames_per_video]

    video_dir = os.path.join(save_root, video_name)
    os.makedirs(video_dir, exist_ok=True)

    for idx in frame_indices:
        versions = {
            'clean': clean[idx],
            'degraded_nearest': lr_upsampled[idx],
            'bilinear_upsample': bilinear[idx],
            'solver_YourUNet': restored[idx],
        }

        for label, tensor in versions.items():
            img_np = tensor.permute(1, 2, 0).cpu().numpy()
            img_uint8 = (np.clip(img_np, 0, 1) * 255).astype(np.uint8)
            img_pil = Image.fromarray(img_uint8)
            save_path = os.path.join(video_dir, f'frame{idx+1:03d}_{label}.png')
            img_pil.save(save_path)

    print(f"Saved {len(frame_indices)} frames × {len(versions)} versions for {video_name}")

print(f"\nAll frames saved to: {save_root}")

Saved 6 frames × 4 versions for baseball
Saved 6 frames × 4 versions for basketball-game
Saved 6 frames × 4 versions for bear
Saved 6 frames × 4 versions for bears-ball
Saved 6 frames × 4 versions for bike-packing

All frames saved to: /content/drive/MyDrive/ResearchProject/benchmark_results_sr/sr_frames


In [10]:
import subprocess

output_dir = '/content/drive/MyDrive/ResearchProject/sr_videos'
os.makedirs(output_dir, exist_ok=True)
fps = 24

for video_name, data in all_sr_results.items():
    clean = data['clean']
    bilinear = data['bilinear_upsampled']
    restored = data['restored']
    T = clean.shape[0]

    videos_to_save = {
        'clean': clean,
        'degraded_nearest': data['lr_upsampled'],
        'bilinear_upsample': bilinear,
        'solver_YourUNet': restored,
    }

    for label, tensor in videos_to_save.items():
        frames_np = [tensor[t].permute(1, 2, 0).cpu().numpy() for t in range(T)]
        h, w = frames_np[0].shape[:2]
        save_path = os.path.join(output_dir, f"{video_name}_sr{scale_factor}x_{label}.mp4")

        cmd = [
            'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
            '-pix_fmt', 'rgb24', '-s', f'{w}x{h}', '-r', str(fps),
            '-i', '-', '-c:v', 'libx264', '-crf', '0',
            '-preset', 'medium', '-pix_fmt', 'yuv444p', save_path
        ]
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE,
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for frame in frames_np:
            proc.stdin.write((np.clip(frame, 0, 1) * 255).astype(np.uint8).tobytes())
        proc.stdin.close()
        proc.wait()

    print(f"Saved 3 videos for {video_name}")

print(f"\nAll videos saved to: {output_dir}")

Saved 3 videos for baseball
Saved 3 videos for basketball-game
Saved 3 videos for bear
Saved 3 videos for bears-ball
Saved 3 videos for bike-packing

All videos saved to: /content/drive/MyDrive/ResearchProject/sr_videos


In [11]:
print("=" * 60)
print(f"SUPER-RESOLUTION {scale_factor}x — SUMMARY")
print("=" * 60)

rows = []
for video_name, data in all_sr_results.items():
    clean = data['clean']
    bilinear = data['bilinear_upsampled']
    restored = data['restored']
    T = clean.shape[0]

    psnr_bi, psnr_sol, ssim_bi, ssim_sol = [], [], [], []
    for t in range(T):
        c = clean[t].permute(1, 2, 0).cpu().numpy()
        b = bilinear[t].permute(1, 2, 0).cpu().numpy()
        r = restored[t].permute(1, 2, 0).cpu().numpy()
        psnr_bi.append(compute_psnr(c, np.clip(b, 0, 1), data_range=1.0))
        psnr_sol.append(compute_psnr(c, np.clip(r, 0, 1), data_range=1.0))
        ssim_bi.append(compute_ssim(c, np.clip(b, 0, 1), data_range=1.0, channel_axis=2))
        ssim_sol.append(compute_ssim(c, np.clip(r, 0, 1), data_range=1.0, channel_axis=2))

    rows.append({
        'Video': video_name,
        'Frames': T,
        'PSNR Bilinear': np.mean(psnr_bi),
        'PSNR Solver': np.mean(psnr_sol),
        'PSNR Gain': np.mean(psnr_sol) - np.mean(psnr_bi),
        'SSIM Bilinear': np.mean(ssim_bi),
        'SSIM Solver': np.mean(ssim_sol),
    })

    print(f"  {video_name}: Bilinear {np.mean(psnr_bi):.2f} → Solver {np.mean(psnr_sol):.2f} dB "
          f"(Δ = {np.mean(psnr_sol)-np.mean(psnr_bi):+.2f})")

df = pd.DataFrame(rows)
print(f"\nOverall average:")
print(f"  Bilinear: {df['PSNR Bilinear'].mean():.2f} dB | Solver: {df['PSNR Solver'].mean():.2f} dB | "
      f"Gain: {df['PSNR Gain'].mean():+.2f} dB")

df.to_csv(benchmark_dir / 'sr_benchmark.csv', index=False)
print(f"\nSaved to {benchmark_dir / 'sr_benchmark.csv'}")

SUPER-RESOLUTION 2x — SUMMARY
  baseball: Bilinear 28.73 → Solver 29.46 dB (Δ = +0.73)


KeyboardInterrupt: 

In [12]:
# Cell 10: Generate Excel benchmark report for SR
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from datetime import datetime

# Compute per-video averages
video_summary = []
for video_name, data in all_sr_results.items():
    clean = data['clean']
    lr_upsampled = data['lr_upsampled']
    bilinear = data['bilinear_upsampled']
    restored = data['restored']
    T = clean.shape[0]

    psnr_nn, ssim_nn = [], []
    psnr_bi, ssim_bi = [], []
    psnr_sol, ssim_sol = [], []

    for t in range(T):
        c = clean[t].permute(1, 2, 0).cpu().numpy()
        nn = lr_upsampled[t].permute(1, 2, 0).cpu().numpy()
        bi = bilinear[t].permute(1, 2, 0).cpu().numpy()
        r = restored[t].permute(1, 2, 0).cpu().numpy()

        psnr_nn.append(compute_psnr(c, np.clip(nn, 0, 1), data_range=1.0))
        psnr_bi.append(compute_psnr(c, np.clip(bi, 0, 1), data_range=1.0))
        psnr_sol.append(compute_psnr(c, np.clip(r, 0, 1), data_range=1.0))
        ssim_nn.append(compute_ssim(c, np.clip(nn, 0, 1), data_range=1.0, channel_axis=2))
        ssim_bi.append(compute_ssim(c, np.clip(bi, 0, 1), data_range=1.0, channel_axis=2))
        ssim_sol.append(compute_ssim(c, np.clip(r, 0, 1), data_range=1.0, channel_axis=2))

    video_summary.append({
        'video': video_name,
        'frames': T,
        'psnr_nearest': np.mean(psnr_nn),
        'psnr_bilinear': np.mean(psnr_bi),
        'psnr_solver': np.mean(psnr_sol),
        'ssim_nearest': np.mean(ssim_nn),
        'ssim_bilinear': np.mean(ssim_bi),
        'ssim_solver': np.mean(ssim_sol),
    })

# --- Create Excel ---
wb = openpyxl.Workbook()
header_font = Font(bold=True, size=12)
title_font = Font(bold=True, size=14)
header_fill = PatternFill(start_color='D5E8F0', end_color='D5E8F0', fill_type='solid')
best_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_cell(ws, row, col, fmt=None, bold=False, fill=None):
    cell = ws.cell(row=row, column=col)
    cell.border = thin_border
    cell.alignment = Alignment(horizontal='center')
    if fmt: cell.number_format = fmt
    if bold: cell.font = header_font
    if fill: cell.fill = fill

methods = ['Nearest', 'Bilinear', 'YourUNet Solver']
method_keys_psnr = ['psnr_nearest', 'psnr_bilinear', 'psnr_solver']
method_keys_ssim = ['ssim_nearest', 'ssim_bilinear', 'ssim_solver']

# ===== Sheet 1: PSNR =====
ws1 = wb.active
ws1.title = 'PSNR Comparison'
ws1.cell(row=1, column=1, value=f'Super-Resolution {scale_factor}x — PSNR Comparison (dB)').font = title_font

headers = ['Video', 'Frames'] + methods + ['Best']
for col, h in enumerate(headers, 1):
    ws1.cell(row=3, column=col, value=h)
    ws1.cell(row=3, column=col).font = header_font
    ws1.cell(row=3, column=col).fill = header_fill
    ws1.cell(row=3, column=col).border = thin_border

for i, v in enumerate(video_summary):
    row = 4 + i
    psnr_vals = {m: v[k] for m, k in zip(methods, method_keys_psnr)}
    best_name = max(psnr_vals, key=psnr_vals.get)

    ws1.cell(row=row, column=1, value=v['video'])
    ws1.cell(row=row, column=2, value=v['frames'])
    for j, (m, k) in enumerate(zip(methods, method_keys_psnr)):
        col = 3 + j
        ws1.cell(row=row, column=col, value=round(v[k], 2))
        fill = best_fill if m == best_name else None
        style_cell(ws1, row, col, '0.00', fill=fill)
    ws1.cell(row=row, column=3 + len(methods), value=round(psnr_vals[best_name], 2))
    style_cell(ws1, row, 1)
    style_cell(ws1, row, 2)
    style_cell(ws1, row, 3 + len(methods), '0.00')

ar = 4 + len(video_summary)
ws1.cell(row=ar, column=1, value='AVERAGE')
for j, k in enumerate(method_keys_psnr):
    ws1.cell(row=ar, column=3+j, value=round(np.mean([v[k] for v in video_summary]), 2))
    style_cell(ws1, ar, 3+j, '0.00', bold=True)
style_cell(ws1, ar, 1, bold=True)
style_cell(ws1, ar, 2, bold=True)

ws1.column_dimensions['A'].width = 22
for c in 'BCDEFG':
    ws1.column_dimensions[c].width = 18

# ===== Sheet 2: SSIM =====
ws2 = wb.create_sheet('SSIM Comparison')
ws2.cell(row=1, column=1, value=f'Super-Resolution {scale_factor}x — SSIM Comparison').font = title_font

for col, h in enumerate(headers, 1):
    ws2.cell(row=3, column=col, value=h)
    ws2.cell(row=3, column=col).font = header_font
    ws2.cell(row=3, column=col).fill = header_fill
    ws2.cell(row=3, column=col).border = thin_border

for i, v in enumerate(video_summary):
    row = 4 + i
    ssim_vals = {m: v[k] for m, k in zip(methods, method_keys_ssim)}
    best_name = max(ssim_vals, key=ssim_vals.get)

    ws2.cell(row=row, column=1, value=v['video'])
    ws2.cell(row=row, column=2, value=v['frames'])
    for j, (m, k) in enumerate(zip(methods, method_keys_ssim)):
        col = 3 + j
        ws2.cell(row=row, column=col, value=round(v[k], 4))
        fill = best_fill if m == best_name else None
        style_cell(ws2, row, col, '0.0000', fill=fill)
    ws2.cell(row=row, column=3 + len(methods), value=round(ssim_vals[best_name], 4))
    style_cell(ws2, row, 1)
    style_cell(ws2, row, 2)
    style_cell(ws2, row, 3 + len(methods), '0.0000')

ar2 = 4 + len(video_summary)
ws2.cell(row=ar2, column=1, value='AVERAGE')
for j, k in enumerate(method_keys_ssim):
    ws2.cell(row=ar2, column=3+j, value=round(np.mean([v[k] for v in video_summary]), 4))
    style_cell(ws2, ar2, 3+j, '0.0000', bold=True)
style_cell(ws2, ar2, 1, bold=True)
style_cell(ws2, ar2, 2, bold=True)

ws2.column_dimensions['A'].width = 22
for c in 'BCDEFG':
    ws2.column_dimensions[c].width = 18

# ===== Sheet 3: PSNR Gain over Bilinear =====
ws3 = wb.create_sheet('PSNR Gain')
ws3.cell(row=1, column=1, value=f'PSNR Improvement over Bilinear Upsample (dB)').font = title_font

h3 = ['Video', 'Nearest', 'Bilinear', 'YourUNet Solver']
for col, h in enumerate(h3, 1):
    ws3.cell(row=3, column=col, value=h)
    ws3.cell(row=3, column=col).font = header_font
    ws3.cell(row=3, column=col).fill = header_fill
    ws3.cell(row=3, column=col).border = thin_border

for i, v in enumerate(video_summary):
    row = 4 + i
    ws3.cell(row=row, column=1, value=v['video'])
    style_cell(ws3, row, 1)

    gain_nn = v['psnr_nearest'] - v['psnr_bilinear']
    gain_bi = 0.00
    gain_sol = v['psnr_solver'] - v['psnr_bilinear']

    ws3.cell(row=row, column=2, value=round(gain_nn, 2))
    ws3.cell(row=row, column=3, value=round(gain_bi, 2))
    ws3.cell(row=row, column=4, value=round(gain_sol, 2))

    # Highlight best (excluding bilinear itself)
    gains = {'Nearest': gain_nn, 'YourUNet Solver': gain_sol}
    best = max(gains, key=gains.get)
    style_cell(ws3, row, 2, '0.00', fill=best_fill if best == 'Nearest' else None)
    style_cell(ws3, row, 3, '0.00')
    style_cell(ws3, row, 4, '0.00', fill=best_fill if best == 'YourUNet Solver' else None)

ar3 = 4 + len(video_summary)
ws3.cell(row=ar3, column=1, value='AVERAGE')
style_cell(ws3, ar3, 1, bold=True)
avg_gain_nn = np.mean([v['psnr_nearest'] - v['psnr_bilinear'] for v in video_summary])
avg_gain_sol = np.mean([v['psnr_solver'] - v['psnr_bilinear'] for v in video_summary])
ws3.cell(row=ar3, column=2, value=round(avg_gain_nn, 2))
ws3.cell(row=ar3, column=3, value=0.00)
ws3.cell(row=ar3, column=4, value=round(avg_gain_sol, 2))
for col in range(2, 5):
    style_cell(ws3, ar3, col, '0.00', bold=True)

ws3.column_dimensions['A'].width = 22
for c in 'BCD':
    ws3.column_dimensions[c].width = 18

# ===== Sheet 4: Notes =====
ws4 = wb.create_sheet('Notes')
ws4.cell(row=1, column=1, value='Super-Resolution Benchmark — Methodology Notes').font = title_font
notes = [
    ('Date:', str(datetime.now())),
    ('Test videos:', str(len(video_summary))),
    ('Processing resolution:', f'{clean.shape[3]}×{clean.shape[2]}'),
    ('Scale factor:', str(scale_factor)),
    ('Solver:', 'Kadkhodaie & Simoncelli (2021)'),
    ('sigma_0:', '0.15'), ('sigma_L:', '0.003'), ('h0:', '0.01'), ('beta:', '0.1'),
    ('', ''),
    ('Metrics:', ''),
    ('PSNR', 'Peak Signal-to-Noise Ratio vs clean ground truth (dB). Higher = better.'),
    ('SSIM', 'Structural Similarity vs clean ground truth (0 to 1). Higher = better.'),
    ('PSNR Gain', 'Improvement over bilinear upsample baseline (dB).'),
    ('', ''),
    ('Method Notes:', ''),
    ('Nearest', 'Nearest-neighbor upsample (adjoint of avg_pool). Blocky artifacts.'),
    ('Bilinear', 'Bilinear interpolation upsample. Smooth but blurry. Standard baseline.'),
    ('YourUNet Solver', 'Kadkhodaie solver using your blind UNet denoiser as implicit prior.'),
    ('', ''),
    ('Green cells', 'indicate the best performer for each video.'),
]
for i, (k, val) in enumerate(notes):
    ws4.cell(row=3+i, column=1, value=k).font = Font(bold=True) if k else Font()
    ws4.cell(row=3+i, column=2, value=val)
ws4.column_dimensions['A'].width = 22
ws4.column_dimensions['B'].width = 75

excel_path = str(benchmark_dir / 'sr_benchmark_comparison.xlsx')
wb.save(excel_path)
print(f"✓ Excel saved: {excel_path}")
print(f"  Sheets: {wb.sheetnames}")
print(f"  Green cells = best performer per video")

✓ Excel saved: /content/drive/MyDrive/ResearchProject/benchmark_results_sr/sr_benchmark_comparison.xlsx
  Sheets: ['PSNR Comparison', 'SSIM Comparison', 'PSNR Gain', 'Notes']
  Green cells = best performer per video
